## TMDB Movie Data Analysis With Apache Spark

### STEP 1: Extract data from API

In this first step, Extract data from the movie dataset API. For the scope of the project, fetch specific movies based on their IDs.

Save the extracted data to 'data/raw_file' 

In [1]:
import requests
import os
import sys
from dotenv import load_dotenv
import datetime
import json

In [2]:
# load environment variables from .env file
load_dotenv()

# load environment variable
TMDB_API_KEY = os.getenv('TMDB_API_KEY')

# set base url of TMDB website
TMDB_BASE_URL = 'https://api.themoviedb.org/3/movie'

In [3]:
import requests

def api_data_extraction(api_key, base_url, data_points):
    """
    This function extracts data from a URL using an API key.
    It returns a list of dictionaries, where each dictionary is the
    JSON data for a specific data point.
    
    Args:
        api_key (str): The API key for authentication.
        base_url (str): The base URL for the API endpoint (e.g., 'https://api.themoviedb.org/3/movie').
        data_points (list): A list of data point identifiers (e.g., movie IDs).

    Returns:
        list: A list of the successfully retrieved JSON data objects (dictionaries).
    """
    if not api_key:
        raise ValueError('API key is required!')

    results = []

    params = {
        'api_key': api_key,
        'append_to_response': 'credits'
    }

    for data_point in data_points:
        try:
            # Construct the full URL with the data point
            full_url = f'{base_url}/{data_point}'
            
            response = requests.get(full_url, params=params)
            
            response.raise_for_status()
            
            data = response.json()
            
            # Check for empty data, assuming an empty dictionary is an error
            if not data:
                print(f'Warning: Empty response for data point {data_point}. Skipping.')
                continue

            results.append(data)
            print(f'movie id {data_point} extracted successfully!')
        
        except requests.exceptions.HTTPError as http_error:
            # Log the specific error and continue to the next data point
            print(f'HTTP error for data point {data_point}: {http_error}')
            print(f'Response content: {http_error.response.text}')
            continue
        except requests.exceptions.Timeout as timeout_error:
            print(f"Timeout error for {data_point}: {timeout_error}")
            continue
        except requests.exceptions.ConnectionError as conn_error:
            print(f"Connection error for {data_point}: {conn_error}")
            continue
        except requests.exceptions.RequestException as req_error:
            print(f"An unexpected requests error occurred for {data_point}: {req_error}")
            continue
        except Exception as e:
            print(f"An unexpected error occurred for {data_point}: {e}")
            continue

    return results

In [4]:
# list of movies to extract
movie_ids = [0, 299534, 19995, 140607, 299536, 597, 135397, 420818, 24428, 168259, 99861,
             284054, 12445, 181808, 330457, 351286, 109445, 321612, 260513]

In [5]:
# perform the data extraction
movie_data = api_data_extraction(TMDB_API_KEY, TMDB_BASE_URL, movie_ids)

HTTP error for data point 0: 404 Client Error: Not Found for url: https://api.themoviedb.org/3/movie/0?api_key=6383d74ea1616df62e4286bc3cb11c82&append_to_response=credits
Response content: {"success":false,"status_code":34,"status_message":"The resource you requested could not be found."}
movie id 299534 extracted successfully!
movie id 19995 extracted successfully!
movie id 140607 extracted successfully!
movie id 299536 extracted successfully!
movie id 597 extracted successfully!
movie id 135397 extracted successfully!
movie id 420818 extracted successfully!
movie id 24428 extracted successfully!
movie id 168259 extracted successfully!
movie id 99861 extracted successfully!
movie id 284054 extracted successfully!
movie id 12445 extracted successfully!
movie id 181808 extracted successfully!
movie id 330457 extracted successfully!
movie id 351286 extracted successfully!
movie id 109445 extracted successfully!
movie id 321612 extracted successfully!
movie id 260513 extracted successfull

In [6]:
# save the data as a raw file in data
def save_extracted_data(data, file_path):
    """
    Saves a list of data to a specified JSON file path.

    Args:
        data (list): A list of dictionaries to be saved.
        file_path (str): The full path to the output JSON file, including the filename.
    """

    directory = os.path.dirname(file_path)

    if directory and not os.path.exists(directory):
        try:
            os.makedirs(directory)
            print(f"Created directory: {directory}")
        except OSError as e:
            print(f"Error creating directory {directory}: {e}")
            return # Exit the function if directory creation fail

    try:
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4)
        print(f"Successfully saved data to {file_path}")
    except IOError as e:
        print(f'Error saving data to JSON file: {e}')
    except Exception as e:
        print(f"An unexpected error occurred while saving: {e}")


In [8]:

save_extracted_data(movie_data, 'data/movies_data_raw.json')

Successfully saved data to data/movies_data_raw.json


### STEP 2 - Spark Data Preprocessing and Cleaning

In this second step, preprocess and cleaning the data set. Explore nested structs and extract relevant data from nested fields.

Save clean data to 'data/**processed_data'

In [9]:
from pyspark.sql import SparkSession

In [10]:
spark = SparkSession.builder.appName('TMDB_movie_project').getOrCreate()

In [11]:
# create DataFrame using extracted movie data in memory

movie_data_df = spark.createDataFrame(movie_data)

In [12]:
movie_data_df.printSchema()

root
 |-- adult: boolean (nullable = true)
 |-- backdrop_path: string (nullable = true)
 |-- belongs_to_collection: map (nullable = true)
 |    |-- key: string
 |    |-- value: long (valueContainsNull = true)
 |-- budget: long (nullable = true)
 |-- credits: map (nullable = true)
 |    |-- key: string
 |    |-- value: array (valueContainsNull = true)
 |    |    |-- element: map (containsNull = true)
 |    |    |    |-- key: string
 |    |    |    |-- value: boolean (valueContainsNull = true)
 |-- genres: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: string
 |    |    |-- value: long (valueContainsNull = true)
 |-- homepage: string (nullable = true)
 |-- id: long (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- origin_country: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- original_language: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- overview: string (nullable = true)
 

**We can see the schema of our data. There are nested structs. Let's explore the nested fields,  and extract the relevant data**

In [180]:
# 1. drop irrelevant columns

movie_data_df = movie_data_df.drop('video', 'poster_path', 'imdb_id', 'backdrop_path', 'original_title', 'homepage', 'adult', 'origin_country')

In [17]:
# evaluate nested columns (JSON)
movie_data_df.select('belongs_to_collection', 'spoken_languages', 'production_countries', 'production_companies',
                     'genres', 'credits').show(1, truncate=True)

+---------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|belongs_to_collection|    spoken_languages|production_countries|production_companies|              genres|             credits|
+---------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
| {backdrop_path ->...|[{name -> English...|[{name -> United ...|[{name -> NULL, i...|[{name -> NULL, i...|{cast -> [{cast_i...|
+---------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
only showing top 1 row



In [18]:
# check and clean columns one-by-one
movie_data_df.select('belongs_to_collection').show(truncate=False)

+------------------------------------------------------------------------+
|belongs_to_collection                                                   |
+------------------------------------------------------------------------+
|{backdrop_path -> NULL, name -> NULL, id -> 86311, poster_path -> NULL} |
|{backdrop_path -> NULL, name -> NULL, id -> 87096, poster_path -> NULL} |
|{backdrop_path -> NULL, name -> NULL, id -> 10, poster_path -> NULL}    |
|{backdrop_path -> NULL, name -> NULL, id -> 86311, poster_path -> NULL} |
|NULL                                                                    |
|{backdrop_path -> NULL, name -> NULL, id -> 328, poster_path -> NULL}   |
|{backdrop_path -> NULL, name -> NULL, id -> 762512, poster_path -> NULL}|
|{backdrop_path -> NULL, name -> NULL, id -> 86311, poster_path -> NULL} |
|{backdrop_path -> NULL, name -> NULL, id -> 9485, poster_path -> NULL}  |
|{backdrop_path -> NULL, name -> NULL, id -> 86311, poster_path -> NULL} |
|{backdrop_path -> NULL, 

Notice that there are nulls in the fields of the nested structs? It is a common issue. There are nulls even though those fields are populated. This problem comes from sparks automatic schema inference. It works, but for some reason fails to parse the data correctly.

There are few ways to go about this. 
- I could define the schema using Spark's StructType and StructField. This is recommended for most cases. You'd have to manually create the Struct for all column. 

- You could save the file as a JSON file, and read the file using spark.read.json(...) which is highly optimized for reading JSON files.

You can also do this, the method i will use(dirty approach):
- I have to break my flow and load the data i saved back into the program again
- Or go through all the columns and create Structs for each.

So I will use the "dirty" approach. I will convert the python list containing the extracted API results into RDD, and convert each dictionary object to a json string, and let spark read the strings.

In [153]:
import json

# convert each line into a json string to keep all the qualities
movies_rdd = spark.sparkContext.parallelize(movie_data).map(lambda x: json.dumps(x))

# convert movies rdd to spark dataframe - which is optimized and keeps all features of a json struct
movie_data_df = spark.read.json(movies_rdd)

**01. process *'belongs_to_collection'* column**

In [154]:
movie_data_df.select('belongs_to_collection').show(truncate=False)
movie_data_df.select('belongs_to_collection').printSchema()

+---------------------------------------------------------------------------------------------------------------+
|belongs_to_collection                                                                                          |
+---------------------------------------------------------------------------------------------------------------+
|{/zuW6fOiusv4X9nnW3paHGfXcSll.jpg, 86311, The Avengers Collection, /yFSIUVTCvgYrpalUktulvk3Gi5Y.jpg}           |
|{/gxnvX9kF7RRUQYvB52dMLPgeJkt.jpg, 87096, Avatar Collection, /3C5brXxnBxfkeKWwA1Fh4xvy4wr.jpg}                 |
|{/4z9ijhgEthfRHShoOvMaBlpciXS.jpg, 10, Star Wars Collection, /22dj38IckjzEEUZwN1tPU5VJ1qq.jpg}                 |
|{/zuW6fOiusv4X9nnW3paHGfXcSll.jpg, 86311, The Avengers Collection, /yFSIUVTCvgYrpalUktulvk3Gi5Y.jpg}           |
|NULL                                                                                                           |
|{/njFixYzIxX8jsn6KMSEtAzi4avi.jpg, 328, Jurassic Park Collection, /qIm2nHXLpBBdMxi8dvfr

Now we can see the data is parsed successfully and all fields are populated. Move on to extract the relevant fields.

In [155]:
# extract collection name from belongs_to_collection column
from pyspark.sql.functions import col

movie_data_df = movie_data_df.withColumn('belongs_to_collection',
    col("belongs_to_collection.name")
)

movie_data_df.select('title', 'belongs_to_collection').show(5)

+--------------------+---------------------+
|               title|belongs_to_collection|
+--------------------+---------------------+
|   Avengers: Endgame| The Avengers Coll...|
|              Avatar|    Avatar Collection|
|Star Wars: The Fo...| Star Wars Collection|
|Avengers: Infinit...| The Avengers Coll...|
|             Titanic|                 NULL|
+--------------------+---------------------+
only showing top 5 rows



**02. process *'genres'* column**

In [156]:
movie_data_df.select('genres').show(3, truncate=False)
movie_data_df.select('genres').printSchema()

+----------------------------------------------------------------------+
|genres                                                                |
+----------------------------------------------------------------------+
|[{12, Adventure}, {878, Science Fiction}, {28, Action}]               |
|[{28, Action}, {12, Adventure}, {14, Fantasy}, {878, Science Fiction}]|
|[{12, Adventure}, {28, Action}, {878, Science Fiction}]               |
+----------------------------------------------------------------------+
only showing top 3 rows

root
 |-- genres: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)



let's extract the name fields from genres. There are more than one genres, so we seperate them with pipe, "|".

In [157]:
from pyspark.sql.functions import concat_ws

# extract genres
movie_data_df = movie_data_df.withColumn(
    'genres',
    concat_ws("|", col('genres.name'))
)

movie_data_df.select('title', 'belongs_to_collection','genres').show(5)


+--------------------+---------------------+--------------------+
|               title|belongs_to_collection|              genres|
+--------------------+---------------------+--------------------+
|   Avengers: Endgame| The Avengers Coll...|Adventure|Science...|
|              Avatar|    Avatar Collection|Action|Adventure|...|
|Star Wars: The Fo...| Star Wars Collection|Adventure|Action|...|
|Avengers: Infinit...| The Avengers Coll...|Adventure|Action|...|
|             Titanic|                 NULL|       Drama|Romance|
+--------------------+---------------------+--------------------+
only showing top 5 rows



**03. Explore *'spoken_languages'* column**

In [158]:
movie_data_df.select('spoken_languages').show(5, truncate=False)
movie_data_df.select('spoken_languages').printSchema()

+------------------------------------------------------------------------------------------------------------------------------------------------+
|spoken_languages                                                                                                                                |
+------------------------------------------------------------------------------------------------------------------------------------------------+
|[{English, en, English}, {Japanese, ja, 日本語}, {Xhosa, xh, }]                                                                                 |
|[{English, en, English}, {Spanish, es, Español}]                                                                                                |
|[{English, en, English}]                                                                                                                        |
|[{English, en, English}, {Xhosa, xh, }]                                                                                 

Extract languages and seperate them by pipes (|)

In [159]:
# extract languages
movie_data_df = movie_data_df.withColumn(
    'spoken_languages',
    concat_ws("|", col('spoken_languages.name'))
)

movie_data_df.select('title', 'belongs_to_collection','genres', 'spoken_languages' ).show(5)

+--------------------+---------------------+--------------------+--------------------+
|               title|belongs_to_collection|              genres|    spoken_languages|
+--------------------+---------------------+--------------------+--------------------+
|   Avengers: Endgame| The Avengers Coll...|Adventure|Science...|     English|日本語||
|              Avatar|    Avatar Collection|Action|Adventure|...|     English|Español|
|Star Wars: The Fo...| Star Wars Collection|Adventure|Action|...|             English|
|Avengers: Infinit...| The Avengers Coll...|Adventure|Action|...|            English||
|             Titanic|                 NULL|       Drama|Romance|English|Français|...|
+--------------------+---------------------+--------------------+--------------------+
only showing top 5 rows



**04. Explore *'production_companies'* column**

In [160]:
movie_data_df.select('production_companies').show(3, truncate=False)
movie_data_df.select('production_companies').printSchema()

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|production_companies                                                                                                                                                                                                                    |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[{420, /hUzeosd33nzE5MCNsZxCGEKTXaQ.png, Marvel Studios, US}]                                                                                                                                                                           |
|[{444, NULL, Dune Entertainment, US}, {574, /8wdmrfGcDx3TJx

In [161]:
# extract and seperate company names with "|"
movie_data_df = movie_data_df.withColumn(
   'production_companies',
    concat_ws("|", col('production_companies.name'))
)

movie_data_df.select('title', 'belongs_to_collection','genres', 'spoken_languages', 'production_companies' ).show(5)

+--------------------+---------------------+--------------------+--------------------+--------------------+
|               title|belongs_to_collection|              genres|    spoken_languages|production_companies|
+--------------------+---------------------+--------------------+--------------------+--------------------+
|   Avengers: Endgame| The Avengers Coll...|Adventure|Science...|     English|日本語||      Marvel Studios|
|              Avatar|    Avatar Collection|Action|Adventure|...|     English|Español|Dune Entertainmen...|
|Star Wars: The Fo...| Star Wars Collection|Adventure|Action|...|             English|Lucasfilm Ltd.|Ba...|
|Avengers: Infinit...| The Avengers Coll...|Adventure|Action|...|            English||      Marvel Studios|
|             Titanic|                 NULL|       Drama|Romance|English|Français|...|Paramount Picture...|
+--------------------+---------------------+--------------------+--------------------+--------------------+
only showing top 5 rows



**05. Explore *'production_countries'* column**

In [162]:
movie_data_df.select('production_countries').show(5, truncate=False)
movie_data_df.select('production_countries').printSchema()

+------------------------------------------------------+
|production_countries                                  |
+------------------------------------------------------+
|[{US, United States of America}]                      |
|[{US, United States of America}, {GB, United Kingdom}]|
|[{US, United States of America}]                      |
|[{US, United States of America}]                      |
|[{US, United States of America}]                      |
+------------------------------------------------------+
only showing top 5 rows

root
 |-- production_countries: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- iso_3166_1: string (nullable = true)
 |    |    |-- name: string (nullable = true)



In [163]:
# extract production country name fields, seperated with |
movie_data_df = movie_data_df.withColumn(
   'production_countries',
    concat_ws("|", col('production_countries.name'))
)

movie_data_df.select('title', 'production_companies', 'production_countries').show(5)

+--------------------+--------------------+--------------------+
|               title|production_companies|production_countries|
+--------------------+--------------------+--------------------+
|   Avengers: Endgame|      Marvel Studios|United States of ...|
|              Avatar|Dune Entertainmen...|United States of ...|
|Star Wars: The Fo...|Lucasfilm Ltd.|Ba...|United States of ...|
|Avengers: Infinit...|      Marvel Studios|United States of ...|
|             Titanic|Paramount Picture...|United States of ...|
+--------------------+--------------------+--------------------+
only showing top 5 rows



**05. Explore credits columns**

This step includes a little bit of a different work. let's dive right into it to understand

In [164]:
movie_data_df.select('credits').toPandas()
movie_data_df.select('credits').printSchema()

root
 |-- credits: struct (nullable = true)
 |    |-- cast: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- adult: boolean (nullable = true)
 |    |    |    |-- cast_id: long (nullable = true)
 |    |    |    |-- character: string (nullable = true)
 |    |    |    |-- credit_id: string (nullable = true)
 |    |    |    |-- gender: long (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- known_for_department: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- order: long (nullable = true)
 |    |    |    |-- original_name: string (nullable = true)
 |    |    |    |-- popularity: double (nullable = true)
 |    |    |    |-- profile_path: string (nullable = true)
 |    |-- crew: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- adult: boolean (nullable = true)
 |    |    |    |-- credit_id: string (nullable = true)
 |

We have to perform some feature engineering in this part. 

'cast', 'cast_size', 'director', 'crew_size'

*create cast column*

In [165]:
from pyspark.sql.functions import transform, slice

movie_data_df.select('credits.cast').printSchema()

root
 |-- cast: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- adult: boolean (nullable = true)
 |    |    |-- cast_id: long (nullable = true)
 |    |    |-- character: string (nullable = true)
 |    |    |-- credit_id: string (nullable = true)
 |    |    |-- gender: long (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- known_for_department: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- order: long (nullable = true)
 |    |    |-- original_name: string (nullable = true)
 |    |    |-- popularity: double (nullable = true)
 |    |    |-- profile_path: string (nullable = true)



In [166]:

movie_data_df = movie_data_df.withColumn(
    'casts',
    concat_ws('|', col('credits.cast.name'))
)

create another feature, cast_size.

In [167]:
movie_data_df.select('credits.cast.name').count()

18

In [168]:
from pyspark.sql.functions import size

movie_data_df = movie_data_df.withColumn(
    'cast_size',
    size(col('credits.cast'))
)

create a new feature for crew_size

In [169]:
movie_data_df = movie_data_df.withColumn(
    'crew_size',
    size(col('credits.crew'))
)

movie_data_df.select('*').toPandas()

,adult,backdrop_path,belongs_to_collection,budget,credits,genres,homepage,id,imdb_id,origin_country,...,spoken_languages,status,tagline,title,video,vote_average,vote_count,casts,cast_size,crew_size
0,False,/7RyHsO4yDXtBv1zUU3mTpHeQ0d5.jpg,The Avengers Collection,356000000,"([(False, 493, Tony Stark / Iron Man, 5e85cd73...",Adventure|Science Fiction|Action,https://www.marvel.com/movies/avengers-endgame,299534,tt4154796,[US],...,English|日本語|,Released,Avenge the fallen.,Avengers: Endgame,False,8.239,26644,Robert Downey Jr.|Chris Evans|Mark Ruffalo|Chr...,105,604
1,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,Avatar Collection,237000000,"([(False, 242, Jake Sully, 5602a8a7c3a36855320...",Action|Adventure|Fantasy|Science Fiction,https://www.avatar.com/movies/avatar,19995,tt0499549,[US],...,English|Español,Released,Enter the world of Pandora.,Avatar,False,7.593,32522,Sam Worthington|Zoe Saldaña|Sigourney Weaver|S...,65,988
2,False,/8BTsTfln4jlQrLXUBquXJ0ASQy9.jpg,Star Wars Collection,245000000,"([(False, 8, Han Solo, 52fe4a959251416c750e713...",Adventure|Action|Science Fiction,http://www.starwars.com/films/star-wars-episod...,140607,tt2488496,[US],...,English,Released,Every generation has a story.,Star Wars: The Force Awakens,False,7.300,19907,Harrison Ford|Mark Hamill|Carrie Fisher|Adam D...,183,259
3,False,/mDfJG3LC3Dqb67AZ52x3Z0jU0uB.jpg,The Avengers Collection,300000000,"([(False, 1, Tony Stark / Iron Man, 54a9cfa292...",Adventure|Action|Science Fiction,https://www.marvel.com/movies/avengers-infinit...,299536,tt4154756,[US],...,English|,Released,Destiny arrives all the same.,Avengers: Infinity War,False,8.236,30861,Robert Downey Jr.|Chris Evans|Chris Hemsworth|...,69,725
4,False,/tupgjqhWx5oieQrdyesO3aclUX9.jpg,None,200000000,"([(False, 412, Jack Dawson, 675afdf316336674f4...",Drama|Romance,https://www.paramountmovies.com/movies/titanic,597,tt0120338,[US],...,English|Français|Deutsch|svenska|Italiano|Pусский,Released,Nothing on Earth could come between them.,Titanic,False,7.905,26252,Leonardo DiCaprio|Kate Winslet|Billy Zane|Kath...,117,262
5,False,/yOCRqvrRrxbs5FYq2pX1KtLJwmR.jpg,Jurassic Park Collection,150000000,"([(False, 10, Owen, 5311015a9251410fe600151d, ...",Action|Adventure|Science Fiction|Thriller,https://www.jurassicworld.com/,135397,tt0369610,[US],...,English,Released,The park has opened.,Jurassic World,False,6.699,20939,Chris Pratt|Bryce Dallas Howard|Irrfan Khan|Vi...,53,425
6,False,/4G7SzRAaXYZ5hYfS05wbTzjv2Tn.jpg,The Lion King (Reboot) Collection,260000000,"([(False, 17, Scar (voice), 59890a509251414bfa...",Adventure|Drama|Family|Animation,https://movies.disney.com/the-lion-king-2019,420818,tt6105098,[US],...,English,Released,The king has returned.,The Lion King,False,7.106,10457,Chiwetel Ejiofor|John Oliver|Donald Glover|Jam...,20,46
7,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,The Avengers Collection,220000000,"([(False, 46, Tony Stark / Iron Man, 52fe4495c...",Science Fiction|Action|Adventure,https://www.marvel.com/movies/the-avengers,24428,tt0848228,[US],...,English|हिन्दी|Pусский,Released,Some assembly required.,The Avengers,False,7.792,32775,Robert Downey Jr.|Chris Evans|Mark Ruffalo|Chr...,112,636
8,False,/cHkhb5A4gQRK6zs6Pv7zorHs8Nk.jpg,The Fast and the Furious Collection,190000000,"([(False, 17, Dominic Toretto, 5431dfd10e0a265...",Action|Crime|Thriller,https://www.uphe.com/movies/furious-7,168259,tt2820852,[US],...,العربية|English|Español|ภาษาไทย,Released,Vengeance hits home.,Furious 7,False,7.200,10912,Vin Diesel|Paul Walker|Jason Statham|Michelle ...,48,225
9,False,/kIBK5SKwgqIIuRKhhWrJn3XkbPq.jpg,The Avengers Collection,365000000,"([(False, 76, Tony Stark / Iron Man, 55e256d29...",Action|Adventure|Science Fiction,https://www.marvel.com/movies/avengers-age-of-...,99861,tt2395427,[US],...,English,Released,A new age has come.,Avengers: Age of Ultron,False,7.271,23660,Robert Downey Jr.|Chris Hemsworth|Mark Ruffalo...,71,644


In [7]:
movies_df.show(1)

+---------------+
|_corrupt_record|
+---------------+
|              [|
+---------------+
only showing top 1 row



make new feature for directors

In [170]:
movie_data_df.select('credits.crew').printSchema()

root
 |-- crew: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- adult: boolean (nullable = true)
 |    |    |-- credit_id: string (nullable = true)
 |    |    |-- department: string (nullable = true)
 |    |    |-- gender: long (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- job: string (nullable = true)
 |    |    |-- known_for_department: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- original_name: string (nullable = true)
 |    |    |-- popularity: double (nullable = true)
 |    |    |-- profile_path: string (nullable = true)



In [171]:
from pyspark.sql.functions import explode, col, filter, collect_list

1. Explode the crew array.
2. Filter only directors.
3. Group back by movie.
4. Collect director names into an array.
5. Join them with | into one string.

In [176]:
# create new directors dataframe and join it to the main df after
directors_df = movie_data_df.withColumn(
    "crew_member", 
    explode(col("credits.crew"))
).filter(
    col("crew_member.job") == "Director"
).groupBy("title"
         ).agg(
    concat_ws("|", collect_list(col("crew_member.name"))).alias("directors"))


# join to main df
movie_data_df = movie_data_df.join(directors_df, on='title', how='left')


root
 |-- title: string (nullable = true)
 |-- adult: boolean (nullable = true)
 |-- backdrop_path: string (nullable = true)
 |-- belongs_to_collection: string (nullable = true)
 |-- budget: long (nullable = true)
 |-- credits: struct (nullable = true)
 |    |-- cast: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- adult: boolean (nullable = true)
 |    |    |    |-- cast_id: long (nullable = true)
 |    |    |    |-- character: string (nullable = true)
 |    |    |    |-- credit_id: string (nullable = true)
 |    |    |    |-- gender: long (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- known_for_department: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- order: long (nullable = true)
 |    |    |    |-- original_name: string (nullable = true)
 |    |    |    |-- popularity: double (nullable = true)
 |    |    |    |-- profile_path: string (nullable = t

In [177]:
# drop credits
movie_data_df = movie_data_df.drop('credits')

In [182]:
movie_data_df.printSchema()

root
 |-- title: string (nullable = true)
 |-- belongs_to_collection: string (nullable = true)
 |-- budget: long (nullable = true)
 |-- genres: string (nullable = false)
 |-- id: long (nullable = true)
 |-- original_language: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- popularity: double (nullable = true)
 |-- production_companies: string (nullable = false)
 |-- production_countries: string (nullable = false)
 |-- release_date: string (nullable = true)
 |-- revenue: long (nullable = true)
 |-- runtime: long (nullable = true)
 |-- spoken_languages: string (nullable = false)
 |-- status: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- vote_average: double (nullable = true)
 |-- vote_count: long (nullable = true)
 |-- casts: string (nullable = false)
 |-- cast_size: integer (nullable = false)
 |-- crew_size: integer (nullable = false)
 |-- directors: string (nullable = true)



#### STEP 3: Fix datatypes and perform transformations

In [185]:
# convert release date to datetime datatype

from pyspark.sql.functions import to_date

movie_data_df = movie_data_df.withColumn(
    'release_date', to_date('release_date', 'yyyy-MM-dd')
)

In [186]:
# convert runtime dtype from long to int
movie_data_df = movie_data_df.withColumn(
    'runtime', col('runtime').cast('int')
)

In [189]:
movie_data_df.select('budget', 'revenue').show(1)

+---------+----------+
|   budget|   revenue|
+---------+----------+
|356000000|2799439100|
+---------+----------+
only showing top 1 row



Convert Budget and Revenue, which are in million to billion

In [195]:
movie_data_df = (
    movie_data_df
    .withColumn("revenue", (col("revenue") / 1_000_000).cast("double"))
    .withColumn("budget", (col("budget") / 1_000_000).cast("double"))
    .withColumnRenamed("revenue", "revenue_million_usd")
    .withColumnRenamed("budget", "budget_million_usd")
)



AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `revenue` cannot be resolved. Did you mean one of the following? [`genres`, `casts`, `id`, `overview`, `runtime`].;
'Project [title#3199, belongs_to_collection#3237, budget_million_usd#4866, genres#3284, id#3183L, original_language#3186, overview#3188, popularity#3189, production_companies#3393, production_countries#3455, release_date#4240, revenue_million_usd#4843, runtime#4263, spoken_languages#3336, status#3197, tagline#3198, vote_average#3201, vote_count#3202L, casts#3504, cast_size#3542, crew_size#3573, directors#3841, cast(('revenue / 1000000) as double) AS revenue#4896]
+- Project [title#3199, belongs_to_collection#3237, budget#4820 AS budget_million_usd#4866, genres#3284, id#3183L, original_language#3186, overview#3188, popularity#3189, production_companies#3393, production_countries#3455, release_date#4240, revenue_million_usd#4843, runtime#4263, spoken_languages#3336, status#3197, 

In [197]:
(
    movie_data_df
    .withColumn("revenue", (col("revenue") / 1_000_000).cast("double")),2)
    .withColumn("budget", round((col("budget") / 1_000_000).cast("double")),2)
    .withColumnRenamed("revenue", "revenue_million_usd")
    .withColumnRenamed("budget", "budget_million_usd")
)

TypeError: type Column doesn't define __round__ method

In [196]:

movie_data_df.select('title', 'genres', 'budget_million_usd', 'revenue_million_usd', 'runtime', 'release_date', 'status').show(5)

+--------------------+--------------------+------------------+-------------------+-------+------------+--------+
|               title|              genres|budget_million_usd|revenue_million_usd|runtime|release_date|  status|
+--------------------+--------------------+------------------+-------------------+-------+------------+--------+
|   Avengers: Endgame|Adventure|Science...|             356.0|          2799.4391|    181|  2019-04-24|Released|
|              Avatar|Action|Adventure|...|             237.0|        2923.706026|    162|  2009-12-15|Released|
|Star Wars: The Fo...|Adventure|Action|...|             245.0|        2068.223624|    136|  2015-12-15|Released|
|Avengers: Infinit...|Adventure|Action|...|             300.0|        2052.415039|    149|  2018-04-25|Released|
|             Titanic|       Drama|Romance|             200.0|        2264.162353|    194|  1997-11-18|Released|
+--------------------+--------------------+------------------+-------------------+-------+------

In [201]:
# reorder columns
movie_data_df = movie_data_df.select('id', 'title', 'tagline', 'release_date', 'genres', 'belongs_to_collection',
'original_language', 'budget_million_usd', 'revenue_million_usd', 'production_companies',
'production_countries', 'vote_count', 'vote_average', 'popularity', 'runtime',
'overview', 'spoken_languages', 'casts', 'cast_size', 'directors', 'crew_size')

movie_data_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- genres: string (nullable = false)
 |-- belongs_to_collection: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- budget_million_usd: double (nullable = true)
 |-- revenue_million_usd: double (nullable = true)
 |-- production_companies: string (nullable = false)
 |-- production_countries: string (nullable = false)
 |-- vote_count: long (nullable = true)
 |-- vote_average: double (nullable = true)
 |-- popularity: double (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- overview: string (nullable = true)
 |-- spoken_languages: string (nullable = false)
 |-- casts: string (nullable = false)
 |-- cast_size: integer (nullable = false)
 |-- directors: string (nullable = true)
 |-- crew_size: integer (nullable = false)



In [205]:
# check for nulls

from pyspark.sql.functions import col, sum

null_counts = movie_data_df.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in movie_data_df.columns
])
null_counts.show()


+---+-----+-------+------------+------+---------------------+-----------------+------------------+-------------------+--------------------+--------------------+----------+------------+----------+-------+--------+----------------+-----+---------+---------+---------+
| id|title|tagline|release_date|genres|belongs_to_collection|original_language|budget_million_usd|revenue_million_usd|production_companies|production_countries|vote_count|vote_average|popularity|runtime|overview|spoken_languages|casts|cast_size|directors|crew_size|
+---+-----+-------+------------+------+---------------------+-----------------+------------------+-------------------+--------------------+--------------------+----------+------------+----------+-------+--------+----------------+-----+---------+---------+---------+
|  0|    0|      0|           0|     0|                    2|                0|                 0|                  0|                   0|                   0|         0|           0|         0|      0

There are only two null values in the belongs_to_collection column - the movies that had no franchise

In [206]:
# fill null values with Standlone, because they have no franchise
movie_data_df = movie_data_df.fillna({"belongs_to_collection": "Standalone"})

movie_data_df.select('belongs_to_collection').show(truncate=False)

+-----------------------------------+
|belongs_to_collection              |
+-----------------------------------+
|The Avengers Collection            |
|Avatar Collection                  |
|Star Wars Collection               |
|The Avengers Collection            |
|Standalone                         |
|Jurassic Park Collection           |
|The Lion King (Reboot) Collection  |
|The Avengers Collection            |
|The Fast and the Furious Collection|
|The Avengers Collection            |
|Black Panther Collection           |
|Harry Potter Collection            |
|Star Wars Collection               |
|Frozen Collection                  |
|Jurassic Park Collection           |
|Frozen Collection                  |
|Standalone                         |
|The Incredibles Collection         |
+-----------------------------------+



In [208]:
def save_processed_data(df, file_path):
    df.write.csv(file_path, header=True, mode="overwrite")


output = 'data/processed_data.csv'

save_processed_data(movie_data_df, output)